In [ ]:
import streamlit as st
import joblib
import numpy as np

st.title(" Construction Stage Cost & Risk Predictor")

# Load models
reg_model = joblib.load("cost_regressor.pkl")
risk_model = joblib.load("risk_classifier.pkl")
stage_encoder = joblib.load("stage_encoder.pkl")
label_encoder = joblib.load("label_encoder.pkl")

# ==============================
# USER INPUTS
# ==============================

project_size = st.number_input("Project Size (sqft)", min_value=100)

total_budget = st.number_input("Total Project Budget (₹)", min_value=100000)

stage_name = st.selectbox(
    "Select Stage",
    stage_encoder.classes_
)

amount_spent = st.number_input("Amount Spent for this Stage (₹)", min_value=0)

# ==============================
# PREDICTION
# ==============================

if st.button("Predict"):

    # Encode stage
    stage_encoded = stage_encoder.transform([stage_name])[0]

    # --------------------------
    # 1️ Regression Prediction
    # --------------------------

    reg_input = np.array([[
        project_size,
        total_budget,
        stage_encoded
    ]])

    predicted_expected_cost = reg_model.predict(reg_input)[0]

    # --------------------------
    # Classification Prediction
    # --------------------------

    class_input = np.array([[
        project_size,
        total_budget,
        stage_encoded,
        amount_spent,
        predicted_expected_cost
    ]])

    risk_prediction_encoded = risk_model.predict(class_input)[0]
    risk_prediction = label_encoder.inverse_transform(
        [risk_prediction_encoded]
    )[0]

    # --------------------------
    # 3️ Display Results
    # --------------------------

    difference = amount_spent - predicted_expected_cost

    st.subheader(" Results")

    st.write(f" Expected Cost: ₹ {predicted_expected_cost:,.2f}")
    st.write(f" Amount Spent: ₹ {amount_spent:,.2f}")
    st.write(f" Difference: ₹ {difference:,.2f}")
    st.write(f" Risk Level: {risk_prediction}")